# Notebook n°1 : Qualité des données

### Problématique

> **Dans quelle mesure la structure du réseau TGV est-elle associée à la régularité des liaisons, et peut-on exploiter cette structure pour améliorer la prédiction des retards ?**

## 1. Objectifs de ce notebook

Ce premier notebook est consacré à l'évaluation de la **qualité des données SNCF** avant leur exploitation dans les analyses statistiques et les modèles prédictifs.

Les objectifs sont les suivants :

- Examiner la structure et les dimensions du jeu de données.
- Identifier les types des différentes variables.
- Quantifier les valeurs manquantes.
- Rechercher les doublons et vérifier l'unicité des observations.
- Contrôler la cohérence des circulations, des annulations et des retards.
- Vérifier les durées moyennes et les pourcentages des causes de retard.
- Identifier et étudier les observations atypiques ou incohérentes.
- Formuler des recommandations pour la préparation des données.

Cette étape permet de limiter les risques de biais et d'erreurs lors des analyses et de la modélisation.

Les fonctions sur le contrôle de la qualité des données sont répertoriés dans le fichier quality.py. Les résultats sont regroupés dans un dictionnaire via report.py. Toutes les fonctions possèdents des tests unitaires.

In [ ]:

"""Chargement des données SNCF depuis l'API pour l'EDA."""

# BytesIO permet de traiter les données téléchargées
# comme un fichier stocké temporairement en mémoire.
from io import BytesIO

# Path permet de manipuler les chemins des fichiers et
# dossiers de manière indépendante du système d'exploitation.
from pathlib import Path

import sys

import pandas as pd

# requests permet d'envoyer des requêtes HTTP, notamment
# pour récupérer des données depuis une API.
import requests

# Path.cwd() récupère le dossier de travail actuel
# dans lequel Python s'exécute.
project_root = Path.cwd()

# Si le dossier courant est "notebooks", cela signifie
# que nous sommes probablement dans le sous-dossier
# notebooks/ de notre projet.
if project_root.name == "notebooks":

    # .parent permet de remonter au dossier parent,
    # donc à la racine du projet IADATA701-PROJET.
    project_root = project_root.parent


# Le dossier src/ contient notre package sncf_tgv.
#
# Par défaut, Python ne sait pas forcément où retrouver
# ce package depuis un notebook.
#
# sys.path est la liste des dossiers dans lesquels
# Python cherche les modules à importer.
#
# insert(0, ...) ajoute le chemin src/ au début de cette liste.
sys.path.insert(0, str(project_root / "src"))

# Maintenant que Python connaît le chemin src/,
# nous pouvons importer notre fonction generate_quality_report.
#
# Cette fonction se trouve dans :
# src/sncf_tgv/eda/report.py
from sncf_tgv.eda.report import generate_quality_report


# Cette URL correspond à l'export CSV complet du jeu
# de données sur la régularité mensuelle des TGV.

API_URL = (
    "https://ressources.data.sncf.com/api/explore/v2.1/"
    "catalog/datasets/regularite-mensuelle-tgv-aqst/exports/csv"
)


# requests.get() envoie une requête HTTP GET à l'API SNCF.
#
# L'objectif est de télécharger le fichier CSV disponible
# à l'adresse API_URL.
#
# timeout=60 fixe un délai maximal de 60 secondes
# pour les opérations réseau concernées.
# Cela évite que la requête reste bloquée indéfiniment.
#
# La réponse HTTP est stockée dans la variable response.
response = requests.get(API_URL, timeout=60)


# raise_for_status() vérifie le statut HTTP de la réponse.
#
# Si l'API répond avec un code d'erreur, par exemple :
# 404 : ressource introuvable
# 500 : erreur du serveur
#
# Une exception est déclenchée pour signaler le problème.
#
# Si la réponse est correcte (HTTP 200), le code continue.
response.raise_for_status()

# response.content contient les données téléchargées
# sous forme d'octets (bytes).
#
# BytesIO(response.content) permet à pandas de lire
# ces octets comme s'il s'agissait d'un fichier.
#
# pd.read_csv() transforme le contenu CSV en DataFrame.
#
# sep=";" précise que les colonnes du CSV sont séparées
# par des points-virgules.
#
# Le DataFrame obtenu est stocké dans la variable data.
data = pd.read_csv(
    BytesIO(response.content),
    sep=";",
)


print("Dimensions :", data.shape)
print("Colonnes :", data.columns.tolist())
print("Types :", data.dtypes)


Dimensions : (12907, 26)
Colonnes : ['date', 'service', 'gare_depart', 'gare_arrivee', 'duree_moyenne', 'nb_train_prevu', 'nb_annulation', 'commentaire_annulation', 'nb_train_depart_retard', 'retard_moyen_depart', 'retard_moyen_tous_trains_depart', 'commentaire_retards_depart', 'nb_train_retard_arrivee', 'retard_moyen_arrivee', 'retard_moyen_tous_trains_arrivee', 'commentaires_retard_arrivee', 'nb_train_retard_sup_15', 'retard_moyen_trains_retard_sup15', 'nb_train_retard_sup_30', 'nb_train_retard_sup_60', 'prct_cause_externe', 'prct_cause_infra', 'prct_cause_gestion_trafic', 'prct_cause_materiel_roulant', 'prct_cause_gestion_gare', 'prct_cause_prise_en_charge_voyageurs']


In [19]:

report = generate_quality_report(data)
print(f"""
Rapport sur la qualité des données:
Nombre de lignes: {report["overview"]["n_rows"]}
Nombre de colonnes: {report["overview"]["n_columns"]}
Doublons : {report["duplicates"]["complete_rows"]}
""")





Rapport sur la qualité des données:
Nombre de lignes: 12907
Nombre de colonnes: 26
Doublons : 0



In [20]:
print("Les valeurs manquantes")
display(report["missing_values"])

Les valeurs manquantes


,missing_count,missing_percentage
date,0,0.000000
service,0,0.000000
gare_depart,0,0.000000
gare_arrivee,0,0.000000
duree_moyenne,0,0.000000
nb_train_prevu,0,0.000000
nb_annulation,0,0.000000
commentaire_annulation,12907,100.000000
nb_train_depart_retard,0,0.000000
retard_moyen_depart,0,0.000000


Nous observons que la plupart des colonnes sont renseignées. Nous avons aucune information sur "commentaire_annulation" et "commentaire_retards_depart" car 100% des données sont manquantes. Il faut encore étudier la colonne "commentaires_retard_arrivee" pour laquelle il manque 95% des valeurs. 

In [21]:
print("Vérification des identifiants")
display(report["identifiers"])

# format de date à respecter AAAA-MM
# doublon_metier : vérifie les doublons sur les colonnes date, service, gare_depart, gare_arrivee

Vérification des identifiants


,regle,nombre_anomalies
0,date_invalide,0
1,service_manquant,0
2,gare_depart_manquant,0
3,gare_arrivee_manquant,0
4,doublon_metier,0


In [22]:
print("Vérification des comptages de trains")
display(report["train_counts"])

"""
    comptage_manquant_ou_invalide        valeurs manquantes ou invalides (NaN, None, pd.NA)
    comptage_negatif                     valeurs négatives
    comptage_non_entier                  valeurs non entières
    annulations_excessives               nombre d'annulations dépasse le nombre de trains prévus
    retards_depart_excessifs             nombre de retards au départ dépasse les trains ayant circulé
    retards_arrivee_excessifs            nombre de retards à l'arrivée dépasse les trains ayant circulé
    retards_15_excessifs                 nombre de retards supérieurs à 15 minutes dépasse les trains ayant circulé
    retards_30_excessifs                 nombre de retards supérieurs à 30 minutes dépasse les trains ayant circulé
    retards_60_excessifs                 nombre de retards supérieurs à 60 minutes dépasse les trains ayant circulé
    seuil_60_superieur_30                nombre de retards supérieurs à 60 minutes dépasse celui des retards supérieurs à 30 minutes
"""

Vérification des comptages de trains


,regle,nombre_anomalies
0,comptage_manquant_ou_invalide,0
1,comptage_negatif,43
2,comptage_non_entier,0
3,annulations_excessives,63
4,retards_depart_excessifs,64
5,retards_arrivee_excessifs,63
6,retards_15_excessifs,65
7,retards_30_excessifs,63
8,retards_60_excessifs,63
9,seuil_60_superieur_30,354


"\n    comptage_manquant_ou_invalide        valeurs manquantes ou invalides (NaN, None, pd.NA)\n    comptage_negatif                     valeurs négatives\n    comptage_non_entier                  valeurs non entières\n    annulations_excessives               nombre d'annulations dépasse le nombre de trains prévus\n    retards_depart_excessifs             nombre de retards au départ dépasse les trains ayant circulé\n    retards_arrivee_excessifs            nombre de retards à l'arrivée dépasse les trains ayant circulé\n    retards_15_excessifs                 nombre de retards supérieurs à 15 minutes dépasse les trains ayant circulé\n    retards_30_excessifs                 nombre de retards supérieurs à 30 minutes dépasse les trains ayant circulé\n    retards_60_excessifs                 nombre de retards supérieurs à 60 minutes dépasse les trains ayant circulé\n    seuil_60_superieur_30                nombre de retards supérieurs à 60 minutes dépasse celui des retards supérieurs à 30

Nous remarquons que le nombres d'anomalies est très similaire selon les différentes règles fixées, autour de 63.

In [23]:
print("Vérification des durées")
display(report["durations"])

"""
    duree_manquante_ou_non_numerique     valeurs manquantes ou non numériques (NaN, None, pd.NA, str, ...)
    duree_infinie                        durée infinie (np.inf, -np.inf)
    trajet_duree_invalide                durée négative
    trajet_duree_non_entiere             durée non entière
    moyenne_trains_retardes_negative     Au moins une moyenne des trains retardés est négative à l'arrivée ou au départ
"""

Vérification des durées


,regle,nombre_anomalies
0,duree_manquante_ou_non_numerique,0
1,duree_infinie,0
2,trajet_duree_invalide,73
3,trajet_duree_non_entiere,0
4,moyenne_trains_retardes_negative,2


"\n    duree_manquante_ou_non_numerique     valeurs manquantes ou non numériques (NaN, None, pd.NA, str, ...)\n    duree_infinie                        durée infinie (np.inf, -np.inf)\n    trajet_duree_invalide                durée négative\n    trajet_duree_non_entiere             durée non entière\n    moyenne_trains_retardes_negative     Au moins une moyenne des trains retardés est négative à l'arrivée ou au départ\n"

Des trajets possèdent des durées négatives ce qui n'est pas cohérent.

In [24]:
print("Vérification des pourcentages")
display(report["percentages"])

Vérification des pourcentages


,regle,nombre_anomalies
0,pourcentage_manquant,0
1,pourcentage_non_numerique,0
2,pourcentage_negatif,0
3,pourcentage_superieur_100,0
4,somme_causes_differente_100,282


La somme de certains pourcentages est différente de 100. Il faudra vérifier si l'écart est import ou de quelques dixiemes.

In [25]:
print("Vérification des commentaires")
display(report["comments"])

Vérification des commentaires


,colonne,nb_manquants,nb_vides,nb_renseignes,pourcentage_renseignes
0,commentaire_annulation,12907,0,0,0.00
1,commentaire_retards_depart,12907,0,0,0.00
2,commentaires_retard_arrivee,12209,0,698,5.41


Comme préalablement remarqué très peu de commentaires sont renseignés.

In [26]:

# Sélectionner les observations dont la durée moyenne
# du trajet est inférieure ou égale à zéro.
durations_invalides = data.loc[
    data["duree_moyenne"] <= 0,
    [
        "date",
        "service",
        "gare_depart",
        "gare_arrivee",
        "duree_moyenne",
        "nb_train_prevu",
        "nb_annulation",
    ],
]

# Compter les durées négatives et nulles séparément.
print(
    "Durées négatives :",
    (data["duree_moyenne"] < 0).sum(),
)

print(
    "Durées nulles :",
    (data["duree_moyenne"] == 0).sum(),
)

display(durations_invalides.head(20))


Durées négatives : 0
Durées nulles : 73


,date,service,gare_depart,gare_arrivee,duree_moyenne,nb_train_prevu,nb_annulation
3406,2020-04,National,PARIS MONTPARNASSE,ST MALO,0,0,8
3407,2020-04,National,PARIS MONTPARNASSE,TOURS,0,0,9
3409,2020-04,National,NANTES,STRASBOURG,0,0,2
3411,2020-04,National,PARIS NORD,DOUAI,0,0,12
3412,2020-04,National,DUNKERQUE,PARIS NORD,0,0,15
3414,2020-04,National,MARSEILLE ST CHARLES,TOURCOING,0,0,1
3415,2020-04,National,LYON PART DIEU,MARNE LA VALLEE,0,0,19
3416,2020-04,National,MARNE LA VALLEE,MARSEILLE ST CHARLES,0,0,14
3417,2020-04,National,MARSEILLE ST CHARLES,MARNE LA VALLEE,0,0,13
3418,2020-04,National,PARIS LYON,AIX EN PROVENCE TGV,0,0,17
